In [19]:
import re
import boto3
import json
import unicodedata
import pandas as pd
from pathlib import Path

In [28]:
s3 = boto3.client("s3")

bucket_name = "proyecto-electricidad-ela"
s3_key = "bronze/ree/ree_raw_2024_2025.json"

response = s3.get_object(Bucket=bucket_name, Key=s3_key)
raw_data = response["Body"].read()

data_all = json.loads(raw_data.decode("utf-8"))

print("Bloques cargados:", len(data_all))
print(type(data_all))
print(data_all[0].keys())

Bloques cargados: 24
<class 'list'>
dict_keys(['year', 'month', 'data'])


In [29]:
lista_total = []

for bloque in data_all:
    year = bloque["year"]
    month = bloque["month"]
    data = bloque["data"]

    for energia in data["included"][0]["attributes"]["content"]:
        nombre = energia["attributes"]["title"]
        valores = energia["attributes"]["values"]

        df_temp = pd.DataFrame(valores)
        df_temp["energia"] = nombre
        df_temp["year"] = year
        df_temp["month"] = month

        lista_total.append(df_temp)

In [30]:
df_esios_long = pd.concat(lista_total, ignore_index=True)

In [31]:
df_esios_long["datetime"] = pd.to_datetime(df_esios_long["datetime"], utc=True, errors="coerce")
df_esios_long["datetime"] = df_esios_long["datetime"].dt.tz_convert("Europe/Madrid").dt.tz_localize(None)

In [32]:
df_esios_long.rename(columns={
    "value": "energia_valor",
    "percentage": "energia_pct",
    "datetime": "fecha"
}, inplace=True)

In [33]:
def normalizar_texto(texto):
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"[^\w\s]", "", texto)
    texto = re.sub(r"\s+", "_", texto)
    return texto

In [34]:
df_esios_long["energia"] = df_esios_long["energia"].apply(normalizar_texto)

In [35]:
print(df_esios_long.head())
print(df_esios_long["energia"].unique())

   energia_valor  energia_pct      fecha     energia  year  month
0      58381.148     0.197189 2024-01-01  hidraulica  2024      1
1      68004.167     0.155704 2024-01-02  hidraulica  2024      1
2      93862.430     0.220157 2024-01-03  hidraulica  2024      1
3     139605.013     0.442881 2024-01-04  hidraulica  2024      1
4      83467.494     0.179563 2024-01-05  hidraulica  2024      1
['hidraulica' 'eolica' 'solar_fotovoltaica' 'solar_termica' 'hidroeolica'
 'otras_renovables' 'residuos_renovables' 'generacion_renovable']


In [36]:
df_esios_wide = df_esios_long.pivot_table(
    index="fecha",
    columns="energia",
    values="energia_valor",
    aggfunc="sum"
).reset_index()

if "generacion_renovable" in df_esios_wide.columns:
    df_esios_wide.rename(columns={
        "generacion_renovable": "generacion_renovable_total"
    }, inplace=True)

columnas_renovables = [
    "hidraulica",
    "eolica",
    "solar_fotovoltaica",
    "solar_termica",
    "hidroeolica",
    "otras_renovables",
    "residuos_renovables"
]

presentes = [c for c in columnas_renovables if c in df_esios_wide.columns]

if "generacion_renovable_total" not in df_esios_wide.columns:
    df_esios_wide["generacion_renovable_total"] = df_esios_wide[presentes].sum(axis=1)

df_esios_wide["fecha"] = pd.to_datetime(df_esios_wide["fecha"], errors="coerce")

df_esios_wide["year"] = df_esios_wide["fecha"].dt.year
df_esios_wide["month"] = df_esios_wide["fecha"].dt.month
df_esios_wide["day"] = df_esios_wide["fecha"].dt.day

df_esios_wide.head()

energia,fecha,eolica,generacion_renovable_total,hidraulica,hidroeolica,otras_renovables,residuos_renovables,solar_fotovoltaica,solar_termica,year,month,day
0,2024-01-01,159629.924,296067.656,58381.148,3.650,6171.128,2470.343,66642.505,2768.958,2024,1,1
1,2024-01-02,307120.989,436751.715,68004.167,2.920,6820.630,2427.951,49615.015,2760.043,2024,1,2
2,2024-01-03,284990.434,426342.548,93862.430,3.037,7570.068,2412.098,36440.805,1063.676,2024,1,3
3,2024-01-04,138827.131,315220.327,139605.013,9.976,8209.237,2548.542,26019.065,1.363,2024,1,4
4,2024-01-05,306634.568,464836.559,83467.494,7.722,8140.782,2421.029,61906.689,2258.275,2024,1,5


In [37]:
df_esios_wide.isnull().sum()

energia
fecha                         0
eolica                        0
generacion_renovable_total    0
hidraulica                    0
hidroeolica                   1
otras_renovables              0
residuos_renovables           0
solar_fotovoltaica            0
solar_termica                 0
year                          0
month                         0
day                           0
dtype: int64

In [38]:
df_esios_wide = df_esios_wide.dropna(subset=["fecha"])
df_esios_wide["year"] = df_esios_wide["fecha"].dt.year
df_esios_wide["month"] = df_esios_wide["fecha"].dt.month
df_esios_wide["day"] = df_esios_wide["fecha"].dt.day
df_esios_wide.isnull().sum()

energia
fecha                         0
eolica                        0
generacion_renovable_total    0
hidraulica                    0
hidroeolica                   1
otras_renovables              0
residuos_renovables           0
solar_fotovoltaica            0
solar_termica                 0
year                          0
month                         0
day                           0
dtype: int64

In [39]:
output_path = Path("../data/silver/esios/esios_diario.parquet")

output_path.parent.mkdir(parents=True, exist_ok=True)

df_esios_wide.to_parquet(output_path, index=False)

In [40]:
print(df_esios_wide["fecha"].min(), df_esios_wide["fecha"].max())
print(df_esios_wide["fecha"].nunique())
print(df_esios_wide["fecha"].isnull().sum())

2024-01-01 00:00:00 2025-12-31 00:00:00
731
0
